# 🏠 HVAC-RL 完整 Pipeline

## 运行说明
1. 确保使用 **GPU 运行时** (Runtime > Change runtime type > GPU)
2. 按顺序运行每个 cell
3. 数据会自动保存到 Google Drive

## 流程
```
PPO训练 → 样本选择 → LLM Rollout → Fine-tuning → 评估
```

---
## Step 1: 环境设置

In [ ]:
# ===== 1.1 挂载 Google Drive =====
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/rl', exist_ok=True)
print("✓ Google Drive 已挂载")

In [ ]:
# ===== 1.2 克隆项目 =====
import os

if not os.path.exists("/content/HAVC-control-with-reinforcement-learning-update"):
    !git clone https://github.com/Mo119m/HAVC-control-with-reinforcement-learning-update.git
    print("✓ 项目已克隆")
else:
    print("✓ 项目已存在")

PROJECT_ROOT = "/content/HAVC-control-with-reinforcement-learning-update"
os.chdir(PROJECT_ROOT)

In [ ]:
# ===== 1.3 安装依赖 =====
!pip install -q stable-baselines3==2.1.0 gymnasium
!pip install -q transformers accelerate bitsandbytes peft
!pip install -q scikit-learn matplotlib tqdm
!pip install -q pvlib  # BEAR 环境需要

print("✓ 依赖安装完成")

In [ ]:
# ===== 1.4 设置 Python 路径 =====
import sys
PROJECT_ROOT = "/content/HAVC-control-with-reinforcement-learning-update"
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# 验证 BEAR 可用
try:
    from BEAR.Env.env_building import BuildingEnvReal
    print("✓ BEAR 环境已加载")
except ImportError as e:
    print(f"✗ BEAR 导入失败: {e}")

In [ ]:
# ===== 1.5 设置数据路径 =====
import os

# 可能的数据路径（按优先级）
POSSIBLE_DATA_PATHS = [
    "/content/drive/MyDrive/rl/Data",
    "/content/drive/MyDrive/HAVC-control-with-reinforcement-learning-update/Data",
    "/content/drive/MyDrive/Data",
]

BEAR_DATA = "/content/HAVC-control-with-reinforcement-learning-update/BEAR/Data"
os.makedirs(BEAR_DATA, exist_ok=True)

# 查找数据源
data_found = False
for data_path in POSSIBLE_DATA_PATHS:
    if os.path.exists(data_path):
        print(f"✓ 找到数据: {data_path}")
        !cp -r "{data_path}"/* {BEAR_DATA}/
        print(f"✓ 数据已复制到 {BEAR_DATA}")
        data_found = True
        break

if not data_found:
    print("⚠ 未找到数据，请将数据上传到以下任一位置:")
    for p in POSSIBLE_DATA_PATHS:
        print(f"   - {p}")

# 显示数据内容
print(f"\nBEAR/Data 目录内容:")
!ls {BEAR_DATA}/

---
## Step 2: PPO 训练

In [ ]:
# ===== 2.1 PPO 训练配置 =====
import os

PPO_CONFIG = {
    "total_timesteps": 200000,  # 可调整
    "building": "OfficeSmall",
    "climate": "Hot_Dry",
    "save_dir": "/content/output_full/01_ppo",
}

os.makedirs(PPO_CONFIG["save_dir"], exist_ok=True)
print(f"配置: {PPO_CONFIG}")

In [ ]:
# ===== 2.2 运行 PPO 训练 =====
import os
os.chdir("/content/HAVC-control-with-reinforcement-learning-update/core_modules")
os.environ["PYTHONPATH"] = "/content/HAVC-control-with-reinforcement-learning-update"

!python ppo_collect.py \
    --building {PPO_CONFIG["building"]} \
    --climate {PPO_CONFIG["climate"]} \
    --total_timesteps {PPO_CONFIG["total_timesteps"]} \
    --output_dir {PPO_CONFIG["save_dir"]}

In [ ]:
# ===== 2.3 保存到 Google Drive =====
import shutil

drive_ppo_dir = "/content/drive/MyDrive/rl/01_ppo"
os.makedirs(drive_ppo_dir, exist_ok=True)

for f in ["ppo_trajectory.json", "training_results.png", "training_metrics.json"]:
    src = f"{PPO_CONFIG['save_dir']}/{f}"
    if os.path.exists(src):
        shutil.copy(src, f"{drive_ppo_dir}/{f}")
        print(f"✓ {f} 已保存")

!ls -la {drive_ppo_dir}

---
## Step 3: 样本选择

In [ ]:
# ===== 3.1 运行样本选择 =====
import os
os.chdir("/content/HAVC-control-with-reinforcement-learning-update/core_modules")

SELECTION_OUTPUT = "/content/output_full/02_few_shot"
os.makedirs(SELECTION_OUTPUT, exist_ok=True)

!python select_representative.py \
    --traj "{PPO_CONFIG['save_dir']}/ppo_trajectory.json" \
    --out_dir {SELECTION_OUTPUT} \
    --preselect 2000 \
    --clusters 12 \
    --n_per_cluster 20 \
    --building "OfficeSmall" \
    --climate "Hot_Dry" \
    --location "Tucson"

In [ ]:
# ===== 3.2 保存到 Google Drive =====
drive_fs_dir = "/content/drive/MyDrive/rl/02_few_shot"
os.makedirs(drive_fs_dir, exist_ok=True)

fs_file = f"{SELECTION_OUTPUT}/few_shot_examples_structured.json"
if os.path.exists(fs_file):
    shutil.copy(fs_file, f"{drive_fs_dir}/few_shot_examples_structured.json")
    print(f"✓ Few-shot 样本已保存")

---
## Step 4: LLM Rollout

In [ ]:
# ===== 4.1 LLM Rollout =====
import os
import sys

sys.path.insert(0, "/content/HAVC-control-with-reinforcement-learning-update")
os.chdir("/content/HAVC-control-with-reinforcement-learning-update/core_modules")

ROLLOUT_OUTPUT = "/content/output_full/03_llm_rollout"
os.makedirs(ROLLOUT_OUTPUT, exist_ok=True)

os.environ["MODEL_NAME"] = "Qwen/Qwen2.5-7B-Instruct"
os.environ["PYTHONPATH"] = "/content/HAVC-control-with-reinforcement-learning-update"

!PYTHONPATH=/content/HAVC-control-with-reinforcement-learning-update python rollout_fewshot_version.py \
    --fewshot_json "{SELECTION_OUTPUT}/few_shot_examples_structured.json" \
    --output "{ROLLOUT_OUTPUT}/llm_trajectory.json" \
    --building "OfficeSmall" \
    --climate "Hot_Dry" \
    --max_steps 200

In [ ]:
# ===== 4.2 保存到 Google Drive =====
drive_rollout_dir = "/content/drive/MyDrive/rl/03_llm_rollout"
os.makedirs(drive_rollout_dir, exist_ok=True)

rollout_file = f"{ROLLOUT_OUTPUT}/llm_trajectory.json"
if os.path.exists(rollout_file):
    shutil.copy(rollout_file, f"{drive_rollout_dir}/llm_trajectory.json")
    print(f"✓ LLM 轨迹已保存")

---
## Step 5: Fine-tuning (LoRA)

In [ ]:
# ===== 5.1 Fine-tuning =====
import os
os.chdir("/content/HAVC-control-with-reinforcement-learning-update/core_modules")

FINETUNE_OUTPUT = "/content/output_full/04_finetune"
os.makedirs(FINETUNE_OUTPUT, exist_ok=True)

!PYTHONPATH=/content/HAVC-control-with-reinforcement-learning-update python 7b_finetune_fixed.py \
    --trajectory_path "{ROLLOUT_OUTPUT}/llm_trajectory.json" \
    --output_dir {FINETUNE_OUTPUT} \
    --model_name "Qwen/Qwen2.5-7B-Instruct" \
    --num_epochs 3 \
    --batch_size 4 \
    --learning_rate 2e-4

In [ ]:
# ===== 5.2 保存到 Google Drive =====
drive_ft_dir = "/content/drive/MyDrive/rl/04_finetune"
os.makedirs(drive_ft_dir, exist_ok=True)

lora_dir = f"{FINETUNE_OUTPUT}/lora_adapter"
if os.path.exists(lora_dir):
    !cp -r {lora_dir} {drive_ft_dir}/
    print(f"✓ LoRA adapter 已保存")

---
## Step 6: 评估对比

In [ ]:
# ===== 6.1 使用 Fine-tuned 模型 Rollout =====
import os
os.chdir("/content/HAVC-control-with-reinforcement-learning-update/core_modules")

EVAL_OUTPUT = "/content/output_full/05_eval"
os.makedirs(EVAL_OUTPUT, exist_ok=True)

!PYTHONPATH=/content/HAVC-control-with-reinforcement-learning-update python 7Blora_rollout.py \
    --lora_path "{FINETUNE_OUTPUT}/lora_adapter" \
    --output "{EVAL_OUTPUT}/finetuned_trajectory.json" \
    --building "OfficeSmall" \
    --climate "Hot_Dry" \
    --max_steps 200

In [ ]:
# ===== 6.2 结果对比 =====
import json
import numpy as np
import matplotlib.pyplot as plt

def load_rewards(path):
    if not os.path.exists(path):
        return None
    with open(path, 'r') as f:
        traj = json.load(f)
    return [step.get('reward', 0) for step in traj]

# 加载数据
ppo_rewards = load_rewards(f"{PPO_CONFIG['save_dir']}/ppo_trajectory.json")
llm_rewards = load_rewards(f"{ROLLOUT_OUTPUT}/llm_trajectory.json")
ft_rewards = load_rewards(f"{EVAL_OUTPUT}/finetuned_trajectory.json")

# 打印统计
print("=" * 50)
print("结果对比")
print("=" * 50)
if ppo_rewards: print(f"PPO Expert:      Mean={np.mean(ppo_rewards):.2f}, Sum={np.sum(ppo_rewards[:200]):.2f}")
if llm_rewards: print(f"LLM Before FT:   Mean={np.mean(llm_rewards):.2f}, Sum={np.sum(llm_rewards):.2f}")
if ft_rewards:  print(f"LLM After FT:    Mean={np.mean(ft_rewards):.2f}, Sum={np.sum(ft_rewards):.2f}")
print("=" * 50)

In [ ]:
# ===== 6.3 可视化对比 =====
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

data = [
    (ppo_rewards[:200] if ppo_rewards else None, 'PPO Expert', 'blue'),
    (llm_rewards, 'LLM Before FT', 'orange'),
    (ft_rewards, 'LLM After FT', 'green'),
]

for rewards, label, color in data:
    if rewards:
        axes[0].plot(np.cumsum(rewards), label=label, color=color, alpha=0.8)
        axes[1].plot(rewards, label=label, color=color, alpha=0.5)

axes[0].set_xlabel('Step')
axes[0].set_ylabel('Cumulative Reward')
axes[0].set_title('Cumulative Reward')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel('Step')
axes[1].set_ylabel('Step Reward')
axes[1].set_title('Step Reward')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/output_full/comparison.png', dpi=150)
plt.show()

---
## 🎉 完成！

所有结果已保存到 Google Drive:
- `/content/drive/MyDrive/rl/01_ppo/` - PPO 训练结果
- `/content/drive/MyDrive/rl/02_few_shot/` - Few-shot 样本
- `/content/drive/MyDrive/rl/03_llm_rollout/` - LLM Rollout 结果
- `/content/drive/MyDrive/rl/04_finetune/` - Fine-tuned 模型

---
## 📋 快速恢复 (Colab 断开后)

In [ ]:
# ===== 快速恢复脚本 =====
# 如果 Colab 断开，运行这个 cell 恢复环境

from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/Mo119m/HAVC-control-with-reinforcement-learning-update.git 2>/dev/null || echo "Already exists"
!pip install -q stable-baselines3 gymnasium transformers accelerate bitsandbytes peft scikit-learn matplotlib pvlib

import sys
import os
sys.path.insert(0, "/content/HAVC-control-with-reinforcement-learning-update")

# 复制数据 - 检查多个可能的路径
BEAR_DATA = "/content/HAVC-control-with-reinforcement-learning-update/BEAR/Data"
os.makedirs(BEAR_DATA, exist_ok=True)

POSSIBLE_DATA_PATHS = [
    "/content/drive/MyDrive/rl/Data",
    "/content/drive/MyDrive/HAVC-control-with-reinforcement-learning-update/Data",
    "/content/drive/MyDrive/Data",
]

for data_path in POSSIBLE_DATA_PATHS:
    if os.path.exists(data_path):
        !cp -r "{data_path}"/* {BEAR_DATA}/
        print(f"✓ 数据已从 {data_path} 复制")
        break

# 恢复输出
!mkdir -p /content/output_full
!cp -r /content/drive/MyDrive/rl/01_ppo /content/output_full/ 2>/dev/null || true
!cp -r /content/drive/MyDrive/rl/02_few_shot /content/output_full/ 2>/dev/null || true
!cp -r /content/drive/MyDrive/rl/03_llm_rollout /content/output_full/ 2>/dev/null || true
!cp -r /content/drive/MyDrive/rl/04_finetune /content/output_full/ 2>/dev/null || true

print("✓ 环境已恢复")
!ls /content/output_full/ 2>/dev/null || echo "输出目录为空，从头开始"